In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%python
customers = spark.table("restaurant_lakehouse.bronze.customers")
restaurants = spark.table("restaurant_lakehouse.bronze.restaurants")
menu_items = spark.table("restaurant_lakehouse.bronze.menu_items")
orders = spark.table("restaurant_lakehouse.bronze.historical_orders")
reviews = spark.table("restaurant_lakehouse.bronze.reviews")

print("Bronze tables loaded successfully")
print("Customers:", customers.count())
print("Restaurants:", restaurants.count())
print("Menu Items:", menu_items.count())
print("Orders:", orders.count())
print("Reviews:", reviews.count())

Bronze tables loaded successfully
Customers: 500
Restaurants: 5
Menu Items: 145
Orders: 8000
Reviews: 83


In [0]:
%python
dim_customer = (
    customers
    .filter(F.col("customer_id").isNotNull())
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("phone", F.col("phone").cast("string"))
    .dropDuplicates(["customer_id"])
)
(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("restaurant_lakehouse.silver.dim_customer")
)

In [0]:
%python
dim_menu_items = (
    menu_items
    .filter(
        F.col("restaurant_id").isNotNull() &
        F.col("item_id").isNotNull()
    )
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("spice_level", F.trim(F.col("spice_level")))
    .dropDuplicates(["restaurant_id", "item_id"])
)
(
    dim_menu_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("restaurant_lakehouse.silver.dim_menu_items")
)

In [0]:
%python
dim_restaurants = (
    restaurants
    .filter(F.col("restaurant_id").isNotNull())
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("address", F.trim(F.col("address")))
    .withColumn("phone", F.trim(F.col("phone")))
    .dropDuplicates(["restaurant_id"])
)
(
    dim_restaurants.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("restaurant_lakehouse.silver.dim_restaurants")
)

In [0]:
%python
display(
    orders.select(
        "order_id",
        "items",
        "total_amount",
        "created_at"
    ).limit(10)
)

order_id,items,total_amount,created_at
ORD-20260125-880072,"[{""item_id"": ""ITEM-103"", ""name"": ""Chicken 65"", ""category"": ""Starter"", ""quantity"": 2, ""unit_price"": 38.24, ""subtotal"": 76.48}, {""item_id"": ""ITEM-102"", ""name"": ""Paneer Tikka"", ""category"": ""Starter"", ""quantity"": 3, ""unit_price"": 36.48, ""subtotal"": 109.44}, {""item_id"": ""ITEM-502"", ""name"": ""Rasmalai (2 pcs)"", ""category"": ""Dessert"", ""quantity"": 1, ""unit_price"": 17.5, ""subtotal"": 17.5}, {""item_id"": ""ITEM-303"", ""name"": ""Lamb Rogan Josh"", ""category"": ""Main Course"", ""quantity"": 2, ""unit_price"": 61.83, ""subtotal"": 123.66}]",327.08,2026-01-25T10:10:47.393Z
ORD-20260125-142045,"[{""item_id"": ""ITEM-102"", ""name"": ""Paneer Tikka"", ""category"": ""Starter"", ""quantity"": 2, ""unit_price"": 33.54, ""subtotal"": 67.08}, {""item_id"": ""ITEM-202"", ""name"": ""Dal Makhani"", ""category"": ""Main Course"", ""quantity"": 2, ""unit_price"": 39.34, ""subtotal"": 78.68}, {""item_id"": ""ITEM-601"", ""name"": ""Masala Chai"", ""category"": ""Beverage"", ""quantity"": 3, ""unit_price"": 11.54, ""subtotal"": 34.62}, {""item_id"": ""ITEM-405"", ""name"": ""Jeera Rice"", ""category"": ""Rice"", ""quantity"": 2, ""unit_price"": 17.89, ""subtotal"": 35.78}]",216.16,2026-01-25T10:22:38.393Z
ORD-20260125-287452,"[{""item_id"": ""ITEM-302"", ""name"": ""Chicken Tikka Masala"", ""category"": ""Main Course"", ""quantity"": 3, ""unit_price"": 49.12, ""subtotal"": 147.36}, {""item_id"": ""ITEM-304"", ""name"": ""Fish Curry"", ""category"": ""Main Course"", ""quantity"": 1, ""unit_price"": 56.56, ""subtotal"": 56.56}, {""item_id"": ""ITEM-501"", ""name"": ""Gulab Jamun (2 pcs)"", ""category"": ""Dessert"", ""quantity"": 2, ""unit_price"": 14.59, ""subtotal"": 29.18}]",233.1,2026-01-25T10:35:21.393Z
ORD-20260125-473010,"[{""item_id"": ""ITEM-102"", ""name"": ""Paneer Tikka"", ""category"": ""Starter"", ""quantity"": 3, ""unit_price"": 33.78, ""subtotal"": 101.34}, {""item_id"": ""ITEM-303"", ""name"": ""Lamb Rogan Josh"", ""category"": ""Main Course"", ""quantity"": 1, ""unit_price"": 67.1, ""subtotal"": 67.1}, {""item_id"": ""ITEM-602"", ""name"": ""Mango Lassi"", ""category"": ""Beverage"", ""quantity"": 3, ""unit_price"": 17.78, ""subtotal"": 53.34}, {""item_id"": ""ITEM-202"", ""name"": ""Dal Makhani"", ""category"": ""Main Course"", ""quantity"": 2, ""unit_price"": 39.37, ""subtotal"": 78.74}]",300.52,2026-01-25T10:40:05.393Z
ORD-20260125-276805,"[{""item_id"": ""ITEM-305"", ""name"": ""Chicken Biryani"", ""category"": ""Main Course"", ""quantity"": 3, ""unit_price"": 47.29, ""subtotal"": 141.87}, {""item_id"": ""ITEM-402"", ""name"": ""Garlic Naan"", ""category"": ""Bread"", ""quantity"": 2, ""unit_price"": 9.74, ""subtotal"": 19.48}]",161.35,2026-01-25T10:43:28.393Z
ORD-20260125-877887,"[{""item_id"": ""ITEM-102"", ""name"": ""Paneer Tikka"", ""category"": ""Starter"", ""quantity"": 3, ""unit_price"": 33.78, ""subtotal"": 101.34}, {""item_id"": ""ITEM-602"", ""name"": ""Mango Lassi"", ""category"": ""Beverage"", ""quantity"": 2, ""unit_price"": 17.78, ""subtotal"": 35.56}, {""item_id"": ""ITEM-503"", ""name"": ""Kulfi"", ""category"": ""Dessert"", ""quantity"": 3, ""unit_price"": 19.79, ""subtotal"": 59.37}, {""item_id"": ""ITEM-104"", ""name"": ""Aloo Tikki Chaat"", ""category"": ""Starter"", ""quantity"": 3, ""unit_price"": 22.02, ""subtotal"": 66.06}, {""item_id"": ""ITEM-305"", ""name"": ""Chicken Biryani"", ""category"": ""Main Course"", ""quantity"": 2, ""unit_price"": 49.29, ""subtotal"": 98.58}]",360.91,2026-01-25T11:04:28.393Z
ORD-20260125-426127,"[{""item_id"": ""ITEM-601"", ""name"": ""Masala Chai"", ""category"": ""Beverage"", ""quantity"": 1, ""unit_price"": 11.59, ""subtotal"": 11.59}, {""item_id"": ""ITEM-302"", ""name"": ""Chicken Tikka Masala"", ""category"": ""Main Course"", ""quantity"": 3, ""unit_price"": 49.12, ""subtotal"": 147.36}, {""item_id"": ""ITEM-305"", ""name"": ""Chicken Biryani"", ""category"

In [0]:
%python
sample_items = (
    orders
    .select("items")
    .filter(F.col("items").isNotNull())
    .limit(5)
    .collect()
)

for row in sample_items:
    print(row["items"])

[{"item_id": "ITEM-103", "name": "Chicken 65", "category": "Starter", "quantity": 2, "unit_price": 38.24, "subtotal": 76.48}, {"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 3, "unit_price": 36.48, "subtotal": 109.44}, {"item_id": "ITEM-502", "name": "Rasmalai (2 pcs)", "category": "Dessert", "quantity": 1, "unit_price": 17.5, "subtotal": 17.5}, {"item_id": "ITEM-303", "name": "Lamb Rogan Josh", "category": "Main Course", "quantity": 2, "unit_price": 61.83, "subtotal": 123.66}]
[{"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 2, "unit_price": 33.54, "subtotal": 67.08}, {"item_id": "ITEM-202", "name": "Dal Makhani", "category": "Main Course", "quantity": 2, "unit_price": 39.34, "subtotal": 78.68}, {"item_id": "ITEM-601", "name": "Masala Chai", "category": "Beverage", "quantity": 3, "unit_price": 11.54, "subtotal": 34.62}, {"item_id": "ITEM-405", "name": "Jeera Rice", "category": "Rice", "quantity": 2, "unit_price": 17

In [0]:
%python
orders_clean = (
    orders
    .filter(F.col("order_id").isNotNull())
    .withColumn(
        "total_amount",
        F.regexp_replace(
            F.col("total_amount"),
            r"[^0-9.-]",
            ""
        ).cast("double")
    )
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    )
    .withColumn(
        "order_type",
        F.trim(F.col("order_type"))
    )
    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method"))
    )
    .withColumn(
        "order_status",
        F.trim(F.col("order_status"))
    )
    .dropDuplicates(["order_id"])
)

In [0]:
%python
fact_orders = orders_clean.select(
    "order_id",
    "timestamp",
    "restaurant_id",
    "customer_id",
    "order_type",
    "total_amount",
    "payment_method",
    "order_status",
    "created_at"
)

In [0]:
%python
orders_clean = (
    orders
    .filter(F.col("order_id").isNotNull())
    .withColumn(
        "total_amount_clean",
        F.regexp_replace(
            F.col("total_amount"),
            r"[^0-9.-]",
            ""
        )
    )
    .withColumn(
        "total_amount",
        F.expr("try_cast(total_amount_clean AS DOUBLE)")
    )
    .drop("total_amount_clean")
    .withColumn(
        "created_at",
        F.expr("try_cast(created_at AS TIMESTAMP)")
    )
    .withColumn(
        "order_type",
        F.trim(F.col("order_type"))
    )
    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method"))
    )
    .withColumn(
        "order_status",
        F.trim(F.col("order_status"))
    )
    .dropDuplicates(["order_id"])
)

In [0]:
%python
bad_total_amount = orders_clean.filter(
    F.col("total_amount").isNull()
).count()

print("Invalid/null total_amount rows:", bad_total_amount)


Invalid/null total_amount rows: 0


In [0]:
%python
fact_orders = orders_clean.select(
    "order_id",
    "timestamp",
    "restaurant_id",
    "customer_id",
    "order_type",
    "total_amount",
    "payment_method",
    "order_status",
    "created_at"
)
(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("restaurant_lakehouse.silver.fact_orders")
)

In [0]:
%python
fact_reviews = (
    reviews
    .filter(F.col("review_id").isNotNull())
    .withColumn("review_text", F.trim(F.col("review_text")))
    .withColumn("rating", F.col("rating").cast("int"))
    .dropDuplicates(["review_id"])
)
fact_reviews = fact_reviews.filter(
    F.col("rating").between(1, 5)
)
(
    fact_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("restaurant_lakehouse.silver.fact_reviews")
)

In [0]:
%sql

SELECT
    r.name AS restaurant_name,
    COUNT(o.order_id) AS total_orders,
    ROUND(SUM(o.total_amount), 2) AS total_revenue,
    ROUND(AVG(o.total_amount), 2) AS average_order_value
FROM restaurant_lakehouse.silver.fact_orders o
JOIN restaurant_lakehouse.silver.dim_restaurants r
    ON o.restaurant_id = r.restaurant_id
GROUP BY r.restaurant_id, r.name
ORDER BY total_revenue DESC;

restaurant_name,total_orders,total_revenue,average_order_value
Spice Route City Centre,1656,296566.16,179.09
Spice Route Downtown,1580,289269.25,183.08
Spice Route Mall of Emirates,1613,283894.1,176.0
Spice Route Al Wahda,1560,282317.11,180.97
Spice Route Marina,1591,280707.84,176.43


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import *

orders = spark.table(
    "restaurant_lakehouse.bronze.historical_orders"
)

print("Orders loaded:", orders.count())

Orders loaded: 8000


In [0]:
%python
sample_json = (
    orders
    .select("items")
    .filter(
        F.col("items").isNotNull() &
        (F.trim(F.col("items")) != "")
    )
    .first()["items"]
)

print(sample_json)
item_schema = spark.range(1).select(
    F.schema_of_json(F.lit(sample_json)).alias("schema")
).first()["schema"]

print(item_schema)

[{"item_id": "ITEM-103", "name": "Chicken 65", "category": "Starter", "quantity": 2, "unit_price": 38.24, "subtotal": 76.48}, {"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 3, "unit_price": 36.48, "subtotal": 109.44}, {"item_id": "ITEM-502", "name": "Rasmalai (2 pcs)", "category": "Dessert", "quantity": 1, "unit_price": 17.5, "subtotal": 17.5}, {"item_id": "ITEM-303", "name": "Lamb Rogan Josh", "category": "Main Course", "quantity": 2, "unit_price": 61.83, "subtotal": 123.66}]
ARRAY<STRUCT<category: STRING, item_id: STRING, name: STRING, quantity: BIGINT, subtotal: DOUBLE, unit_price: DOUBLE>>


In [0]:
%python
fact_orders_base = spark.table(
    "restaurant_lakehouse.silver.fact_orders"
)

orders_with_items = (
    fact_orders_base
    .join(
        orders.select("order_id", "items"),
        on="order_id",
        how="left"
    )
    .withColumn(
        "parsed_items",
        F.from_json(
            F.col("items"),
            item_schema
        )
    )
)

In [0]:
%python
from pyspark.sql import functions as F

orders = spark.table(
    "restaurant_lakehouse.bronze.historical_orders"
)

orders.select(
    "order_id",
    "items"
).show(10, truncate=False)
row = (
    orders
    .filter(F.col("items").isNotNull())
    .select("items")
    .first()
)

print(repr(row["items"]))

+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|order_id           |items                                                                                                                                                                                                                                                                                                

In [0]:
%python
orders_clean = (
    orders
    .filter(F.col("order_id").isNotNull())
    .withColumn(
        "total_amount_clean",
        F.regexp_replace(
            F.col("total_amount"),
            r"[^0-9.-]",
            ""
        )
    )
    .withColumn(
        "total_amount",
        F.expr(
            "try_cast(total_amount_clean AS DOUBLE)"
        )
    )
    .drop("total_amount_clean")
    .withColumn(
        "created_at",
        F.expr(
            "try_cast(created_at AS TIMESTAMP)"
        )
    )
    .withColumn(
        "order_type",
        F.trim(F.col("order_type"))
    )
    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method"))
    )
    .withColumn(
        "order_status",
        F.trim(F.col("order_status"))
    )
    .dropDuplicates(["order_id"])
)

In [0]:
%python
fact_orders = orders_clean.select(
    "order_id",
    "timestamp",
    "restaurant_id",
    "customer_id",
    "order_type",
    "total_amount",
    "payment_method",
    "order_status",
    "created_at"
)

(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "restaurant_lakehouse.silver.fact_orders"
    )
)
schema_string = (
    spark
    .range(1)
    .select(
        F.schema_of_json(
            F.lit(sample_json)
        ).alias("schema")
    )
    .first()["schema"]
)

print(schema_string)
sample_json = (
    orders
    .filter(
        F.col("items").isNotNull() &
        (F.trim(F.col("items")) != "")
    )
    .select("items")
    .first()["items"]
)

print(sample_json)
schema_string = (
    spark
    .range(1)
    .select(
        F.schema_of_json(
            F.lit(sample_json)
        ).alias("schema")
    )
    .first()["schema"]
)

print(schema_string)

ARRAY<STRUCT<category: STRING, item_id: STRING, name: STRING, quantity: BIGINT, subtotal: DOUBLE, unit_price: DOUBLE>>
[{"item_id": "ITEM-103", "name": "Chicken 65", "category": "Starter", "quantity": 2, "unit_price": 38.24, "subtotal": 76.48}, {"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 3, "unit_price": 36.48, "subtotal": 109.44}, {"item_id": "ITEM-502", "name": "Rasmalai (2 pcs)", "category": "Dessert", "quantity": 1, "unit_price": 17.5, "subtotal": 17.5}, {"item_id": "ITEM-303", "name": "Lamb Rogan Josh", "category": "Main Course", "quantity": 2, "unit_price": 61.83, "subtotal": 123.66}]
ARRAY<STRUCT<category: STRING, item_id: STRING, name: STRING, quantity: BIGINT, subtotal: DOUBLE, unit_price: DOUBLE>>


In [0]:
%python
orders_with_items = (
    orders
    .withColumn(
        "parsed_items",
        F.from_json(
            F.col("items"),
            schema_string
        )
    )
)


In [0]:
%python
orders = spark.table(
    "restaurant_lakehouse.bronze.historical_orders"
)

raw_item = (
    orders
    .filter(F.col("items").isNotNull())
    .select("items")
    .first()["items"]
)

print("NORMAL:")
print(raw_item)

print("\nREPR:")
print(repr(raw_item))

print("\nLENGTH:")
print(len(raw_item))

NORMAL:
[{"item_id": "ITEM-103", "name": "Chicken 65", "category": "Starter", "quantity": 2, "unit_price": 38.24, "subtotal": 76.48}, {"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 3, "unit_price": 36.48, "subtotal": 109.44}, {"item_id": "ITEM-502", "name": "Rasmalai (2 pcs)", "category": "Dessert", "quantity": 1, "unit_price": 17.5, "subtotal": 17.5}, {"item_id": "ITEM-303", "name": "Lamb Rogan Josh", "category": "Main Course", "quantity": 2, "unit_price": 61.83, "subtotal": 123.66}]

REPR:
'[{"item_id": "ITEM-103", "name": "Chicken 65", "category": "Starter", "quantity": 2, "unit_price": 38.24, "subtotal": 76.48}, {"item_id": "ITEM-102", "name": "Paneer Tikka", "category": "Starter", "quantity": 3, "unit_price": 36.48, "subtotal": 109.44}, {"item_id": "ITEM-502", "name": "Rasmalai (2 pcs)", "category": "Dessert", "quantity": 1, "unit_price": 17.5, "subtotal": 17.5}, {"item_id": "ITEM-303", "name": "Lamb Rogan Josh", "category": "Main Course", "quan

In [0]:
%python
raw_csv = spark.read.text(
    "/Volumes/restaurant_lakehouse/landing/restaurant_files/historical_orders.csv"
)

raw_csv.show(3, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                             

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%python
fixed_orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .csv(
        "/Volumes/restaurant_lakehouse/landing/restaurant_files/historical_orders.csv"
    )
)

In [0]:
%python
fixed_orders.select(
    "order_id",
    "items",
    "total_amount",
    "payment_method",
    "order_status",
    "created_at"
).show(5, truncate=False)

+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------+------------+--------------------------+
|order_id           |items                                                                                                                                                                                                                                                                                                                                                                              

In [0]:
%python
(
    fixed_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "restaurant_lakehouse.bronze.historical_orders"
    )
)

In [0]:
%python
orders = spark.table(
    "restaurant_lakehouse.bronze.historical_orders"
)

In [0]:
%python
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- items: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)



In [0]:
%python
orders_clean = (
    orders
    .filter(F.col("order_id").isNotNull())
    .withColumn(
        "total_amount",
        F.expr("try_cast(total_amount AS DOUBLE)")
    )
    .withColumn(
        "created_at",
        F.expr("try_cast(created_at AS TIMESTAMP)")
    )
    .withColumn(
        "timestamp",
        F.expr("try_cast(timestamp AS TIMESTAMP)")
    )
    .withColumn(
        "order_type",
        F.trim("order_type")
    )
    .withColumn(
        "payment_method",
        F.trim("payment_method")
    )
    .withColumn(
        "order_status",
        F.trim("order_status")
    )
    .dropDuplicates(["order_id"])
)

In [0]:
%python
fact_orders = orders_clean.select(
    "order_id",
    "timestamp",
    "restaurant_id",
    "customer_id",
    "order_type",
    "total_amount",
    "payment_method",
    "order_status",
    "created_at"
)

In [0]:
%python
(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "restaurant_lakehouse.silver.fact_orders"
    )
)

In [0]:
%python
orders_with_items = (
    orders
    .withColumn(
        "parsed_items",
        F.from_json(
            F.col("items"),
            item_schema
        )
    )
)

In [0]:
%python
orders.select("items").show(2, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|items                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [0]:
%python
item_schema_ddl = """
ARRAY<STRUCT<
    item_id: STRING,
    name: STRING,
    category: STRING,
    quantity: INT,
    unit_price: DOUBLE,
    subtotal: DOUBLE
>>
"""

print(item_schema_ddl)


ARRAY<STRUCT<
    item_id: STRING,
    name: STRING,
    category: STRING,
    quantity: INT,
    unit_price: DOUBLE,
    subtotal: DOUBLE
>>



In [0]:
%python
orders_with_items = (
    orders
    .withColumn(
        "parsed_items",
        F.from_json(
            F.col("items"),
            item_schema_ddl
        )
    )
)

In [0]:
%python
orders_with_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- items: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- parsed_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |    |    |-- subtotal: double (nullable = true)



In [0]:
%python
display(
    orders_with_items.select(
        "order_id",
        "parsed_items"
    ).limit(10)
)

order_id,parsed_items
ORD-20260125-880072,"List(List(ITEM-103, Chicken 65, Starter, 2, 38.24, 76.48), List(ITEM-102, Paneer Tikka, Starter, 3, 36.48, 109.44), List(ITEM-502, Rasmalai (2 pcs), Dessert, 1, 17.5, 17.5), List(ITEM-303, Lamb Rogan Josh, Main Course, 2, 61.83, 123.66))"
ORD-20260125-142045,"List(List(ITEM-102, Paneer Tikka, Starter, 2, 33.54, 67.08), List(ITEM-202, Dal Makhani, Main Course, 2, 39.34, 78.68), List(ITEM-601, Masala Chai, Beverage, 3, 11.54, 34.62), List(ITEM-405, Jeera Rice, Rice, 2, 17.89, 35.78))"
ORD-20260125-287452,"List(List(ITEM-302, Chicken Tikka Masala, Main Course, 3, 49.12, 147.36), List(ITEM-304, Fish Curry, Main Course, 1, 56.56, 56.56), List(ITEM-501, Gulab Jamun (2 pcs), Dessert, 2, 14.59, 29.18))"
ORD-20260125-473010,"List(List(ITEM-102, Paneer Tikka, Starter, 3, 33.78, 101.34), List(ITEM-303, Lamb Rogan Josh, Main Course, 1, 67.1, 67.1), List(ITEM-602, Mango Lassi, Beverage, 3, 17.78, 53.34), List(ITEM-202, Dal Makhani, Main Course, 2, 39.37, 78.74))"
ORD-20260125-276805,"List(List(ITEM-305, Chicken Biryani, Main Course, 3, 47.29, 141.87), List(ITEM-402, Garlic Naan, Bread, 2, 9.74, 19.48))"
ORD-20260125-877887,"List(List(ITEM-102, Paneer Tikka, Starter, 3, 33.78, 101.34), List(ITEM-602, Mango Lassi, Beverage, 2, 17.78, 35.56), List(ITEM-503, Kulfi, Dessert, 3, 19.79, 59.37), List(ITEM-104, Aloo Tikki Chaat, Starter, 3, 22.02, 66.06), List(ITEM-305, Chicken Biryani, Main Course, 2, 49.29, 98.58))"
ORD-20260125-426127,"List(List(ITEM-601, Masala Chai, Beverage, 1, 11.59, 11.59), List(ITEM-302, Chicken Tikka Masala, Main Course, 3, 49.12, 147.36), List(ITEM-305, Chicken Biryani, Main Course, 1, 47.29, 47.29), List(ITEM-604, Fresh Lime Soda, Beverage, 1, 12.19, 12.19), List(ITEM-203, Palak Paneer, Main Course, 1, 42.03, 42.03))"
ORD-20260125-633662,"List(List(ITEM-105, Chicken Seekh Kebab, Starter, 3, 42.96, 128.88), List(ITEM-102, Paneer Tikka, Starter, 2, 36.48, 72.96))"
ORD-20260125-784808,"List(List(ITEM-406, Vegetable Biryani, Rice, 3, 41.04, 123.12))"
ORD-20260125-378532,"List(List(ITEM-601, Masala Chai, Beverage, 3, 12.05, 36.15), List(ITEM-402, Garlic Naan, Bread, 3, 10.25, 30.75), List(ITEM-504, Gajar Halwa, Dessert, 1, 22.51, 22.51), List(ITEM-404, Tandoori Roti, Bread, 1, 6.15, 6.15))"


In [0]:
%python
exploded_items = (
    orders_with_items
    .select(
        "order_id",
        "restaurant_id",
        F.explode("parsed_items").alias("item")
    )
)

In [0]:
%python
display(
    exploded_items.select(
        "order_id",
        "restaurant_id",
        "item.*"
    ).limit(20)
)

order_id,restaurant_id,item_id,name,category,quantity,unit_price,subtotal
ORD-20260125-880072,REST-AUH-001,ITEM-103,Chicken 65,Starter,2,38.24,76.48
ORD-20260125-880072,REST-AUH-001,ITEM-102,Paneer Tikka,Starter,3,36.48,109.44
ORD-20260125-880072,REST-AUH-001,ITEM-502,Rasmalai (2 pcs),Dessert,1,17.5,17.5
ORD-20260125-880072,REST-AUH-001,ITEM-303,Lamb Rogan Josh,Main Course,2,61.83,123.66
ORD-20260125-142045,REST-DXB-001,ITEM-102,Paneer Tikka,Starter,2,33.54,67.08
ORD-20260125-142045,REST-DXB-001,ITEM-202,Dal Makhani,Main Course,2,39.34,78.68
ORD-20260125-142045,REST-DXB-001,ITEM-601,Masala Chai,Beverage,3,11.54,34.62
ORD-20260125-142045,REST-DXB-001,ITEM-405,Jeera Rice,Rice,2,17.89,35.78
ORD-20260125-287452,REST-SHJ-001,ITEM-302,Chicken Tikka Masala,Main Course,3,49.12,147.36
ORD-20260125-287452,REST-SHJ-001,ITEM-304,Fish Curry,Main Course,1,56.56,56.56


In [0]:
%python
fact_order_items = (
    exploded_items
    .select(
        "order_id",
        "restaurant_id",
        F.col("item.item_id").alias("item_id"),
        F.col("item.name").alias("item_name"),
        F.col("item.category").alias("category"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.unit_price").alias("unit_price"),
        F.col("item.subtotal").alias("subtotal")
    )
)

In [0]:
%python
(
    fact_order_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "restaurant_lakehouse.silver.fact_order_items"
    )
)

In [0]:
%python
item_schema_ddl = "ARRAY<STRUCT<item_id:STRING,name:STRING,category:STRING,quantity:INT,unit_price:DOUBLE,subtotal:DOUBLE>>"

In [0]:
%sql

SELECT
    item_name,
    category,
    SUM(quantity) AS units_sold,
    ROUND(SUM(subtotal), 2) AS revenue
FROM restaurant_lakehouse.silver.fact_order_items
GROUP BY
    item_name,
    category
ORDER BY revenue DESC
LIMIT 20;

item_name,category,units_sold,revenue
Lamb Rogan Josh,Main Course,1663,109298.58
Fish Curry,Main Course,1657,88853.67
Chicken Biryani,Main Course,1788,86438.17
Chicken Tikka Masala,Main Course,1720,84305.33
Butter Chicken,Main Course,1605,83979.31
Paneer Butter Masala,Main Course,1759,78899.13
Malai Kofta,Main Course,1664,72582.93
Vegetable Biryani,Rice,1663,70301.52
Chicken Seekh Kebab,Starter,1649,70224.66
Palak Paneer,Main Course,1605,67885.38
